# REST API Exercise

A REST API (Representational State Transfer Application Programming Interface) allows structured access to data over HTTP. Unlike web scraping, APIs provide a defined, stable interface with structured responses (typically JSON), making data retrieval more reliable and efficient. Many public institutions and organizations offer open APIs to share data without requiring authentication.

This notebook retrieves economic indicators for Germany from the **World Bank Open Data API** — a free, publicly accessible REST API that requires no API key.

Use Python, `requests`, and `pandas` to retrieve data from a REST API:

## Import Libraries

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'requests'

## Define the Target API

The World Bank Open Data API provides free access to economic indicators for countries worldwide. No API key is required.

- **Base URL:** `https://api.worldbank.org/v2/`
- **Country code:** `DE` (Germany)
- **Indicators used:**
  - `NY.GDP.PCAP.CD` – GDP per capita (current USD)
  - `SL.UEM.TOTL.ZS` – Unemployment rate (% of total labor force)
  - `SP.POP.TOTL` – Total population
- **Documentation:** https://datahelpdesk.worldbank.org/knowledgebase/articles/889392

In [ ]:
BASE_URL = "https://api.worldbank.org/v2/country/DE/indicator/{indicator}"

INDICATORS = {
    "NY.GDP.PCAP.CD": "GDP_per_capita_USD",
    "SL.UEM.TOTL.ZS": "Unemployment_rate_pct",
    "SP.POP.TOTL":    "Population"
}

## Send Requests to the API

Each indicator is retrieved with a separate GET request. The API returns paginated JSON responses — `per_page=100` ensures all years are returned in a single request. The response status code is checked before processing.

In [ ]:
def fetch_indicator(indicator_code, label):
    url = BASE_URL.format(indicator=indicator_code)
    params = {
        "format": "json",
        "per_page": 100,
        "date": "1993:2025"
    }

    response = requests.get(url, params=params)
    print(f"[{label}] Status Code: {response.status_code}", end=" — ")

    if response.status_code == 200:
        print("OK")
        data = response.json()
        return data[1]  # data[0] = metadata, data[1] = records
    else:
        print("FAILED")
        return []

raw_results = {}
for code, label in INDICATORS.items():
    raw_results[label] = fetch_indicator(code, label)

## Parse the JSON Response

The API returns a list of records, each containing the year (`date`) and the value. Records with missing values (`None`) are kept at this stage and handled in the structuring step.

In [ ]:
def parse_records(records, value_column):
    parsed = []
    for record in records:
        year = int(record["date"])
        value = record["value"]
        parsed.append({"Year": year, value_column: value})
    return parsed

# Show a sample of raw JSON for one indicator
print("Sample raw JSON record:")
print(raw_results["GDP_per_capita_USD"][0])

## Identify the Data to be Retrieved

I want to retrieve three economic indicators for Germany covering the years 1993 to 2025 from the World Bank Open Data API: GDP per capita (in USD), the unemployment rate (in percent), and the total population. These indicators will be combined into a single structured dataset to complement the beer consumption and inflation data from the other exercises, enabling a broader economic analysis.

## Extract and Combine Data

Each indicator is parsed into a separate DataFrame and then merged on the `Year` column into a single dataset covering 1993–2025.

In [ ]:
dfs = []
for label, records in raw_results.items():
    parsed = parse_records(records, label)
    df_temp = pd.DataFrame(parsed)
    dfs.append(df_temp)

# Merge all indicators on Year
df_api = dfs[0]
for df_temp in dfs[1:]:
    df_api = df_api.merge(df_temp, on="Year", how="outer")

df_api = df_api.sort_values(by="Year").reset_index(drop=True)

print(f"Data successfully extracted. Found rows: {len(df_api)}")
df_api.head(10)

## Store Data in a Structured Format

Give a brief overview of the data collected (e.g. count, fields, ...).

In [ ]:
print("--- Overview of API data ---")
print(f"Number of rows (years): {df_api.shape[0]}")
print(f"Column names: {list(df_api.columns)}")
print(f"Year range: {df_api['Year'].min()} – {df_api['Year'].max()}")
print(f"Missing values per column:")
print(df_api.isnull().sum())
print()
print("Data Structure Overview:")
df_api.info()

## Visualize Data

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 12))

axes[0].plot(df_api['Year'], df_api['GDP_per_capita_USD'], marker='o', color='steelblue', linewidth=2)
axes[0].set_title('GDP per Capita in Germany (1993–2025)', fontsize=13)
axes[0].set_ylabel('USD')
axes[0].grid(True, linestyle='--', alpha=0.7)

axes[1].plot(df_api['Year'], df_api['Unemployment_rate_pct'], marker='o', color='tomato', linewidth=2)
axes[1].set_title('Unemployment Rate in Germany (1993–2025)', fontsize=13)
axes[1].set_ylabel('Percent (%)')
axes[1].grid(True, linestyle='--', alpha=0.7)

axes[2].plot(df_api['Year'], df_api['Population'], marker='o', color='seagreen', linewidth=2)
axes[2].set_title('Total Population in Germany (1993–2025)', fontsize=13)
axes[2].set_ylabel('People')
axes[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Save the Data

In [ ]:
df_api.to_csv('world_bank_data.csv', index=False)
print("Data successfully saved as 'world_bank_data.csv'")